# 02 — Online learning (D1 rework, Pair C)

Marker's D1 critique on online learning was 30/50: "one model, no warmup, too little data to test, 200 events!". This notebook is the artifact-only view of the rework. Three things to surface:

1. **4-variant prequential comparison** — static / contextual ε-greedy / non-contextual ε-greedy / logistic bandit on the SAME 2000-event stream (`reports/online_learning_results.csv`).
2. **Multi-drift stream** — 2000 events with two planted query-style drifts at events 800 and 1500; ADWIN recalibrated to `delta=0.002` against the longer-window noise floor.
3. **Cold-start warmup** — every variant cold-starts at the AutoML-winning `hybrid_weight` (read from `configs/winning_runcard.yaml`), addressing the "no warmup" critique.

Run the full 4-variant comparison end-to-end with:
```
python -m csai415.online
```
or with custom horizon / drift points:
```
python -m csai415.online --n-events 2000 --drift-points 800 1500
```
Re-running this notebook just loads + displays the artifacts that command produces (no retraining).

In [ ]:
import os
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

if Path.cwd().name == 'notebooks':
    os.chdir('..')

## 1 — 4-variant comparison

Rows are the 4 learner variants; columns are the per-segment mean NDCG@5 (pre-drift, between drift 1 and drift 2, after drift 2) plus the ADWIN firing count. The static row is the AutoML-winning fixed weight — the boring baseline every other row is compared against.

In [ ]:
results = pd.read_csv('reports/online_learning_results.csv')
results

## 2 — Prequential plot

Rolling-mean NDCG@5 (window=50) per variant on the same 2000-event stream. Red dashed lines mark the two planted drifts. The static baseline drops at each drift; the adaptive variants should recover.

In [ ]:
display(Image('reports/prequential.png'))

## 3 — Headline claim

Acceptance bar (§6.C): "online learner improves NDCG@5 by ≥ 5% vs static after the drift point." With three bandit variants we check the bar three ways.

In [ ]:
static_post1 = results.set_index('model').loc['static', 'post_drift_1_ndcg5']
for model in ['eps_greedy_contextual', 'eps_greedy_noncontext', 'logistic_bandit']:
    lift = (results.set_index('model').loc[model, 'post_drift_1_ndcg5'] - static_post1) / (static_post1 + 1e-9) * 100
    verdict = 'PASS (>=5%)' if lift >= 5 else 'MISS'
    print(f"  {model:<28} vs static (post-drift-1): {lift:+6.1f}%   {verdict}")

ctx  = results.set_index('model').loc['eps_greedy_contextual',  'post_drift_1_ndcg5']
nctx = results.set_index('model').loc['eps_greedy_noncontext', 'post_drift_1_ndcg5']
ctx_lift = (ctx - nctx) / (nctx + 1e-9) * 100
print(f"\n  contextual vs non-contextual (do features help?): {ctx_lift:+.1f}%")